# JavaScript Unit Testing with Jest

A comprehensive guide to writing, organizing, and measuring unit tests in JavaScript using the Jest testing framework. Covers everything from your first test through mocks, async testing, and code coverage.

## Table of Contents

- [Module 01: Writing Your First Unit Test with Jest](#module-01-writing-your-first-unit-test-with-jest)
- [Module 02: Exploring Common Matchers](#module-02-exploring-common-matchers)
- [Module 03: Using Mocks](#module-03-using-mocks)
- [Module 04: Testing Asynchronous Code](#module-04-testing-asynchronous-code)
- [Module 05: Measuring Code Coverage](#module-05-measuring-code-coverage)

In [ ]:
from pathlib import Path
from IPython.display import HTML, display
import base64

def render_diagram(image_path, width=900):
    data = base64.b64encode(Path(image_path).read_bytes()).decode()
    display(HTML(f'<div style="text-align:center"><img src="data:image/png;base64,{data}" width="{width}"/></div>'))

## Module 01: Writing Your First Unit Test with Jest

An introduction to the Jest testing framework — why unit tests matter, how to install and configure Jest, and how to structure test files with hooks and `describe` blocks.

### Version Requirements

This course uses:
- JavaScript 2022 (fully applicable to JS 2017–2022)
- Jest v29
- Node.js v18
- npm v8

### Why Write Unit Tests?

- Manual testing takes too much time and cannot scale
- Unit tests run automatically, giving fast feedback on every change
- Provide confidence in the quality of source code
- Facilitate **Test-Driven Development (TDD)**
- Act as living documentation of expected behavior

### The Test Pyramid

The Test Pyramid shows the recommended distribution of test types. Unit tests sit at the base — they are the most numerous, fastest, and cheapest to run. Integration tests validate interactions between components. E2E (end-to-end) tests cover full workflows but are slowest and most brittle. The arrow indicates **decreasing scope** as you move up.

In [ ]:
render_diagram(Path("assets/01-test-pyramid.png"))

### Key Benefits of Using Jest

- **Zero configuration** required to get started
- Highly intuitive, fluent API
- Runs tests **in parallel** for speed
- Works with the vast majority of JavaScript projects
- Excellent documentation and large community

### Installing Jest

```bash
npm install --save-dev jest
```

Then add a test script to `package.json`:

```json
{
  "scripts": {
    "test": "jest"
  }
}
```

Run all tests with:

```bash
npm test
```

### Jest Configuration Files

Jest auto-discovers configuration from any of these files:

- `jest.config.js` / `.ts` / `.mjs` / `.cjs` / `.json`
- A `"jest"` key inside `package.json`
- An explicit path via `--config`

The config file must return (or export) a configuration object. Example `jest.config.json`:

```json
{
  "testEnvironment": "node"
}
```

### Jest CLI Commands

| Command | Effect |
|---|---|
| `jest` | Runs all tests |
| `jest <file name>` | Runs only the specified test file(s) |
| `jest /tickets/*` | Runs tests matching a glob pattern |
| `jest --findRelatedTests <file>` | Runs tests that execute code in the specified source files |
| `jest -o` / `jest --onlyChanged` | Runs tests related to source files changed since the last commit |
| `npm test -- --onlyChanged` | Same as above, via npm |

### Anatomy of a Jest Test

A basic test file uses `describe` to group related tests, `test` (or `it`) to define individual test cases, and `expect` with a matcher to assert the outcome.

```javascript
const basket = require("../../js/basket/basket");

describe("calculateTotal", () => {
    test("calculates total basket price when no discount applied", () => {
        // Arrange
        const items = [ /* ... */ ];

        // Act
        const total = basket.calculateTotal(items);

        // Assert
        expect(total).toBeCloseTo(2770.00, 2);
    });
});
```

**Hooks** reduce duplication by running setup/teardown code around tests:

```javascript
describe("calculateTotal", () => {
    let items = [];

    beforeEach(() => {
        items = [ /* reset to known state */ ];
    });

    afterEach(() => { /* optional cleanup */ });
    beforeAll(() => { /* run once before all tests in describe */ });
    afterAll(() => { /* run once after all tests in describe */ });

    test("...", () => { /* test using items */ });
});
```

### Demo: Writing Your First Unit Test

A GloboTicket Node.js web application with a basket, events, and promotions. The `after/` snapshot has the completed tests from this module.

See [`01-writing-first-unit-test-demos/`](01-writing-first-unit-test-demos/) for the full project.

Key test file: `tests/basket/basket.test.js`

```javascript
describe("calculateTotal", () => {
    let events = [];
    let items = [];

    beforeEach(() => {
        events = [
            new Event(1, "A Night At The Proms", 2500.00, 2500, 2500),
            new Event(2, "Taylor Swift", 50.00, 5500, 2500),
        ];
        items = [
            new BasketItem(events[0], 1),
            new BasketItem(events[1], 4),
        ];
    });

    test("calculates total basket price when no discount applied", () => {
        const total = basket.calculateTotal(items);
        expect(total).toBeCloseTo(2700.00, 2);
    });

    test("calculates total basket price with discount", () => {
        const total = basket.calculateTotal(items, 800);
        expect(total).toBeCloseTo(1900.00, 2);
    });
});
```

### Module 01 Summary

- **Unit tests** prevent regression bugs, enable TDD, and document expected behavior
- **Jest** requires no configuration to get started; works with any JS project
- A test has three parts: arrange, act, assert — with `describe` for grouping and `test` for individual cases
- **Hooks** (`beforeEach`, `afterEach`, `beforeAll`, `afterAll`) keep tests DRY by centralizing setup and teardown
- The **Jest CLI** supports targeted runs: by filename, glob, or changed files

---

## Module 02: Exploring Common Matchers

Matchers are the assertion building blocks of Jest — functions that check expected versus actual values. This module covers equality, pattern matching, truthiness, and error matchers.

### What Are Matchers?

- Functions that act as **assertions** inside `expect()`
- Fluent and readable: `expect(getUnicorn()).toBeUndefined()`
- Full reference: https://jestjs.io/docs/expect#methods

### Testing Equality: `toBe` vs `toEqual` vs `toStrictEqual`

| Matcher | What it checks |
|---|---|
| `toBe` | Strict reference equality (`===`). Use for primitives. |
| `toEqual` | Deep structural equality. Ignores `undefined` properties and sparse array slots. |
| `toStrictEqual` | Deep equality with strictness: checks keys with `undefined` values, array sparseness, and object types. |

```javascript
// toBe — for primitives
expect(2 + 2).toBe(4);

// toEqual — for objects/arrays (ignores undefined keys)
expect({ a: undefined, b: 1 }).toEqual({ b: 1 }); // passes

// toStrictEqual — stricter object comparison
expect({ a: undefined, b: 1 }).toStrictEqual({ b: 1 }); // FAILS
expect([2, ,]).toStrictEqual([2, undefined]); // FAILS (sparseness differs)
```

### Pattern Matching: `toContain` and `toMatch`

Pattern matching is useful when:
- You only care about **part** of a large object or string
- The full return value does not matter
- The return value is unknown or dynamic

| Matcher | Use case |
|---|---|
| `toContain` | Checks whether an array contains an item, or a string contains a substring |
| `toMatch` | Tests a string against a regular expression or substring |

```javascript
// toContain — in arrays
expect(['apple', 'banana', 'cherry']).toContain('banana');

// toContain — in strings
expect('hello world').toContain('world');

// toMatch — regex
const referralCode = promotions.generateReferralCode('1234');
expect(referralCode).toMatch(/#FRIEND-#\d+-#1234/);

// toMatch — substring
expect(referralCode).toMatch('1234');
```

### Testing Truthiness and Falsiness

JavaScript has exactly **6 falsy values**:

| Value | Type |
|---|---|
| `false` | Boolean |
| `0` | Number |
| `''` | String |
| `null` | Null |
| `undefined` | Undefined |
| `NaN` | Number |

Everything else is truthy.

**Matchers:**

```javascript
// Broad truthiness checks
expect(someValue).toBeTruthy();  // passes for any truthy value
expect(someValue).toBeFalsy();   // passes for any falsy value

// More specific matchers (preferred when precision matters)
expect(value).toBeNull();
expect(value).toBeUndefined();
expect(value).toBeDefined();     // not undefined
expect(value).toBe(0);           // exact zero
```

### Testing Errors: `toThrow`

Use `toThrow` to assert that a function throws an error. **The function must be wrapped in a lambda** — if you call it directly, Jest catches the error before it can assert on it.

```javascript
// Must wrap in an arrow function
expect(() => throwingFunction()).toThrow();

// Assert on error message (string or regex)
expect(() => throwingFunction()).toThrow('Expected error message');
expect(() => throwingFunction()).toThrow(/invalid/i);

// Assert on error type
expect(() => throwingFunction()).toThrow(TypeError);
```

### Demo: Common Matchers

The GloboTicket project adds tests using all of the matchers above.

See [`02-common-matchers-demos/`](02-common-matchers-demos/) for the full project.

Key test file: `tests/promotions/promotions.test.js` — demonstrates `toContain`, `toMatch`, and regex patterns on a generated referral code.

### Module 02 Summary

- **`toBe`** — strict `===` equality; for primitives
- **`toEqual`** — deep structural equality; for objects and arrays
- **`toStrictEqual`** — like `toEqual` but also checks `undefined` properties, array sparseness, and object types
- **`toContain` / `toMatch`** — partial/pattern matching when full value is unknown or irrelevant
- **Truthiness**: 6 falsy values in JS; use `toBeTruthy`/`toBeFalsy` for broad checks, specific matchers for precision
- **`toThrow`** — always wrap the call in an arrow function

---

## Module 03: Using Mocks

Mocks replace real dependencies with controlled stand-ins, isolating the unit under test from slow, unreliable, or complex external code.

### Why Use Mocks?

- **Test links between code** — verify how one unit calls another
- **Capture function calls** — check whether a function was called, with what arguments, and how many times
- **Control return values** — return predictable static data instead of hitting real APIs or databases
- **Avoid slow and unreliable tasks** — network calls, file I/O, timers
- **Isolate the unit under test** — only test one unit at a time

### Creating Simple Mock Functions with `jest.fn()`

`jest.fn()` creates a mock function. You can provide an implementation inline, configure return values, and then assert on how it was called.

```javascript
// Basic mock function
const searchPredicateMock = jest.fn(e => e.ticketPrice < 30);

// Use it in a call
const filterResults = getEvents(events, searchPredicateMock);

// Assert it was called
expect(searchPredicateMock).toHaveBeenCalled();
expect(searchPredicateMock).toHaveBeenCalledTimes(7);
expect(searchPredicateMock).toHaveBeenCalledWith(events[0]);
```

**Configuring return values:**

```javascript
const mockFn = jest.fn();
mockFn.mockReturnValue('static value');       // always returns this
mockFn.mockReturnValueOnce('first call');     // only for the next call
mockFn.mockResolvedValue({ data: [] });       // returns a resolved promise
```

**Spies** wrap an existing function instead of replacing it entirely:

```javascript
const randomMock = jest.spyOn(global.Math, 'random').mockReturnValue(76567);

const referralCode = promotions.generateReferralCode(234);

expect(referralCode).toBe('#FRIEND-#567-#234');
expect(randomMock).toHaveBeenCalled();

// Restore original implementation after test
randomMock.mockRestore();
```

### Automock

Automock tells Jest to automatically mock every imported module in your test files — no explicit mock setup required.

**Enable in `jest.config.js`:**

```javascript
module.exports = {
  automock: true
};
```

**Key behaviours:**

- All imported modules are automatically mocked
- Built-in Node.js modules (e.g. `fs`, `path`) are **not** mocked
- Use `jest.unmock('module-name')` to use the real implementation for a specific module
- Helps ensure tests are truly isolated (only testing one unit)
- Can be hard to introduce into existing test suites (requires auditing all imports)

```javascript
// With automock on, all dependencies come back as jest.fn() stubs
const { getEvents } = require("../../js/events/search"); // auto-mocked
jest.unmock("../../js/events/event");                     // use real Event class
```

### Manual Mocks

Manual mocks live in `__mocks__/` directories alongside the modules they replace. Jest picks them up automatically when you call `jest.mock('module-name')`.

**Directory structure:**

```
js/
  events/
    search.js
__mocks__/
  events/
    search.js    ← manual mock for search.js
```

**When to use manual mocks:**
- When you need a complex, reusable mock that many test files share
- When the auto-generated automock doesn't provide enough control
- For third-party modules (placed in `__mocks__/` at the root next to `node_modules/`)

```javascript
// __mocks__/events/search.js
const getEvents = jest.fn(() => [
    { id: 1, name: 'Mock Event', ticketPrice: 10.00 }
]);
module.exports = { getEvents };
```

```javascript
// In your test file
jest.mock('../../js/events/search');
const { getEvents } = require('../../js/events/search'); // now uses manual mock
```

### Demo: Using Mocks

The GloboTicket project uses `jest.fn()` for a search predicate mock and `jest.spyOn()` to control `Math.random()`.

See [`03-using-mocks-demos/`](03-using-mocks-demos/) for the full project.

Key test files:
- `tests/events/search.test.js` — `jest.fn()` for predicate function
- `tests/promotions/promotions.test.js` — `jest.spyOn()` for `Math.random`

### Module 03 Summary

- **Mock functions** (`jest.fn()`) stub out functionality with static data to keep tests fast and deterministic
- **Spies** (`jest.spyOn()`) wrap existing functions to observe calls without fully replacing them
- **Automock** automatically mocks all imports; enabled in config; use `jest.unmock()` to opt out
- **Manual mocks** live in `__mocks__/` and are shared reusable stubs; activated with `jest.mock('path')`

---

## Module 04: Testing Asynchronous Code

Asynchronous code is increasingly common in JavaScript — network requests, database queries, timers — and requires special handling in Jest to ensure assertions run after the async operation completes.

### Why Async Testing Needs Special Care

If Jest doesn't know a test is async, it finishes the test synchronously before the async work completes — making the test pass vacuously (no assertions ran). You must either **return** or **await** the async work.

### Testing Promises

**Option 1 — return the promise** (Jest waits for it to resolve/reject):

```javascript
test("resolves with user data", () => {
    return expect(getUser(1)).resolves.toEqual({ id: 1, name: 'Alice' });
});

test("rejects when user not found", () => {
    return expect(getUser(-1)).rejects.toThrow('Not found');
});
```

**Option 2 — async/await** (more readable):

```javascript
test("resolves with user data", async () => {
    const user = await getUser(1);
    expect(user).toEqual({ id: 1, name: 'Alice' });
});

test("rejects when user not found", async () => {
    await expect(getUser(-1)).rejects.toThrow('Not found');
});
```

> **Common mistake:** forgetting `return` or `await` causes the test to pass even if the promise rejects.

### Testing Callbacks

For callback-based APIs, use the `done` parameter. Jest will not finish the test until `done()` is called.

```javascript
test("fetches events via callback", (done) => {
    fetchEvents((err, events) => {
        try {
            expect(err).toBeNull();
            expect(events).toHaveLength(3);
            done(); // signal Jest the test is complete
        } catch (error) {
            done(error); // pass assertion errors to done() so Jest records the failure
        }
    });
});
```

> Without the `try/catch`, a failing `expect()` throws an uncaught error inside the callback — `done()` is never called and the test times out instead of reporting the real failure.

### Demo: Testing Asynchronous Code

See [`04-async-testing-demos/`](04-async-testing-demos/) for the full project.

Key test file: `tests/promotions/exchange.test.js` — tests a currency exchange function that returns a promise.

### Module 04 Summary

- Async code is increasingly common and reduces overall task execution time
- **Promises**: must be `return`ed or `await`ed; use `.resolves`/`.rejects` matchers or async/await
- **Callbacks**: use the `done` parameter; always wrap assertions in `try/catch` and call `done(error)` on failure
- Forgetting `return`/`await`/`done` leads to false positives — tests pass without running assertions

---

## Module 05: Measuring Code Coverage

Code coverage measures how much of your source code is actually executed by your tests — a useful indicator of test quality, though not a complete guarantee of correctness.

### What Is Code Coverage?

- A measure of how much **source code is executed** by your test suite
- A good indicator of project quality — but **not perfect** (100% coverage doesn't mean 0 bugs)
- Jest uses **Babel** by default to instrument code for coverage measurement
- Four coverage metrics:

| Metric | Measures |
|---|---|
| **Statements** | Individual executable statements |
| **Branches** | Each branch of conditional logic (`if`/`else`, ternary, `switch`) |
| **Functions** | Each function/method |
| **Lines** | Lines of source code |

### Coverage Example: Branches

The `calculatePrice` function below has two branches — the `if (isSaleItem)` path and the `else` path. To achieve 100% **branch coverage**, your tests must cover both.

A test that only calls `calculatePrice(true)` covers the `if` branch but leaves the `else` branch uncovered — branch coverage would be 50%.

In [ ]:
render_diagram(Path("assets/02-code-coverage-example.png"))

### Collecting Coverage Statistics

**Run with coverage flag:**

```bash
jest --coverage
# or via npm:
npm test -- --coverage
```

**Enable by default in `jest.config.js`:**

```javascript
module.exports = {
  collectCoverage: true,
  collectCoverageFrom: ['js/**/*.js']   // which files to instrument
};
```

Jest generates a **text summary** in the terminal and an **HTML report** in `coverage/lcov-report/index.html`. Open the HTML report to see line-by-line which code paths are covered (green) and which are not (red).

### Configuring Coverage Thresholds

Thresholds enforce a minimum coverage level — the build fails if coverage drops below them. This prevents coverage from silently regressing over time.

```javascript
// jest.config.js
module.exports = {
  collectCoverage: true,
  coverageThreshold: {
    global: {
      statements: 80,
      branches: 75,
      functions: 80,
      lines: 80
    }
  }
};
```

When any threshold is violated, Jest exits with a non-zero status code, failing your CI pipeline.

### Demo: Measuring Code Coverage

See [`05-code-coverage-demos/`](05-code-coverage-demos/) for the full project. This snapshot includes a `jest.config.json` that enables coverage collection.

Install dependencies and run coverage:

```bash
cd 05-code-coverage-demos
npm install
npm test -- --coverage
```

### Module 05 Summary

- **Code coverage** = amount of source code executed by tests; measured across statements, branches, functions, and lines
- Run with `jest --coverage` or configure `collectCoverage: true` in `jest.config.js`
- The HTML report in `coverage/lcov-report/` shows line-by-line coverage
- **Coverage thresholds** (`coverageThreshold`) enforce a minimum level and fail the build if breached

---

### Course Summary

| Topic | Key Points |
|---|---|
| Unit tests | Prevent regressions; enable TDD; act as documentation |
| Jest setup | Zero config to start; powerful CLI for targeted runs |
| Common matchers | `toBe`, `toEqual`, `toStrictEqual`, `toContain`, `toMatch`, `toThrow` |
| Mocks & spies | `jest.fn()`, `jest.spyOn()`, automock, manual mocks in `__mocks__/` |
| Async tests | Promises need `return`/`await`; callbacks need `done` + `try/catch` |
| Code coverage | 4 metrics; HTML report; enforce with thresholds in config |